### ICI nous allons explorer 2 des plus fondamentales techniques du NLP
* CountVectorizer
* Term Frequency-Inverse Document Frequency

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import numpy as np
import scipy.sparse



In [2]:
data = 'cleaned_spam.csv'
df = pd.read_csv(data)
# le fillna() c'est une fonction qui va remplace les NaN qui existent dnas nos colonnes avec ce que nous voulons
df['processed_message'] = df['processed_message'].fillna('')
df.head()

,label,message,tokens,processed_message
0,ham,go until jurong point crazy available only in ...,"['go', 'jurong', 'point', 'crazi', 'avail', 'b...",go jurong point crazi avail bugi n great world...
1,ham,ok lar joking wif u oni,"['ok', 'lar', 'joke', 'wif', 'u', 'oni']",ok lar joke wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"['free', 'entri', '2', 'wkli', 'comp', 'win', ...",free entri 2 wkli comp win fa cup final tkt 21...
3,ham,u dun say so early hor u c already then say,"['u', 'dun', 'say', 'earli', 'hor', 'u', 'c', ...",u dun say earli hor u c alreadi say
4,ham,nah i dont think he goes to usf he lives aroun...,"['nah', 'dont', 'think', 'goe', 'usf', 'live',...",nah dont think goe usf live around though


On passe la premiere technique qui est la BoW [c'est un model qui va convertir les texts aka str en number (parce que les models ne comprennent que les nombres ici)]

Par example,
* Ce message "win free money today" va devenir un tableau de [1, 1, 1, 1].
* Ce message "free money free" va devenir un tableau de [2, 1, 0, 0].

In [3]:
#initialiser la vectorisation avec la fonction CountVectorizer
cv = CountVectorizer()

#maintenenat on va passer au remplissable pour transformer nos str en int ou numeral
X_bow = cv.fit_transform(df['processed_message'])

print("----------------------affichage du shape de notre maxtixe-------------------------")
print("Matrix Shape:")
print(X_bow.shape)

print("----------------------afficher les premiers 20 uniques mots que notre vectoor a appris ")
print("\nFirst 20 Vocabulary Words:")
print(cv.get_feature_names_out()[:20])


----------------------affichage du shape de notre maxtixe-------------------------
Matrix Shape:
(5572, 8065)
----------------------afficher les premiers 20 uniques mots que notre vectoor a appris 

First 20 Vocabulary Words:
['008704050406' '0089mi' '0121' '01223585236' '01223585334' '0125698789'
 '02' '020603' '0207' '02070836089' '02072069400' '02073162414'
 '02085076972' '020903' '021' '050703' '0578' '06' '060505' '061104']


### DTM (La matrice document-terme)
Il est important de comprendre que nous ne créons aucune nouvelle information. Nous prenons simplement les mêmes messages textuels que nous avons déjà et nous les réorganisons sous la forme d'une immense grille de nombres, afin que l'ordinateur puisse les comprendre et les traiter.

Dans le domaine du ML (apprentissage automatique), cette grille est appelée Document-Term Matrix (DTM), ou matrice document-terme en français.

Tu peux l'imaginer comme un immense tableau Excel :
- Chaque ligne représente un message texte.
- Chaque colonne représente un mot de notre vocabulaire global.
- Chaque cellule contient un nombre qui indique combien de fois un mot apparaît dans un message donné.

In [4]:
print(X_bow.toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


### TF-IDF

un peu plus hat=ut nous avons vu loe BoW qui est bien mais presente une faille neanmoins: il considere tous les mots de meme maniere si un mot se repete 1 millions de fois il va continuer par compter aussi ce qui fera que la detection des spams sera plus hard

c'est la que le TF-IDF intervient pour resoudre ce souci en attribuant un score en fonction de leur importance plutot que de compter les frequences
Alors il combine 2 mesures distinctes
- Term Frequency (TF) — Fréquence du terme [Le TF mesure la fréquence à laquelle un terme apparaît dans un document spécifique.]
- Inverse Document Frequency (IDF) — Fréquence documentaire inverse [L'IDF mesure à quel point un mot est rare dans l'ensemble du jeu de données (Si un mot apparaît dans presque tous les messages, son score IDF est faible.
Si un mot apparaît dans très peu de messages, son score IDF est élevé.)]

Pour le calculer on utilise la formule suivante:
### TF-IDF(t,d) = TD(t, d) * IDF(t)
Où :
- \(t\) représente le terme (mot).
- \(d\) représente le document (message).
- TF mesure la fréquence du mot dans le message.
- IDF mesure la rareté du mot dans l'ensemble des documents.

In [6]:
#initialisation toujours avec un elimite de 3000 mots importnat de vocalaire
tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['processed_message'])

print("----------------------affichage du shape de notre maxtixe-------------------------")
print("TF-IDF Matrix Shape:")
print(X.shape)

print("----------------------afficher les premiers 20 uniques mots que notre vectoor a appris ")
print("\nFirst 20 TF-IDF Vocabulary Words:")
print(tfidf.get_feature_names_out()[:20])

----------------------affichage du shape de notre maxtixe-------------------------
TF-IDF Matrix Shape:
(5572, 3000)
----------------------afficher les premiers 20 uniques mots que notre vectoor a appris 

First 20 TF-IDF Vocabulary Words:
['020603' '02073162414' '020903' '0776xxxxxxx' '07786200117' '07821230901'
 '07xxxxxxxxx' '0800' '08000407165' '08000776320' '08000839402'
 '08000930705' '08000938767' '08001950382' '08002986906' '0808' '0845'
 '08452810073' '0870' '08701417012']


##### On voit la difference entre le BoW et le TF-IDF avec les 20 premiers mots les plus importants de notre voc

### On passe maintenant a L'Encodage de la variable cible (Encoding the Target Label)

A cette etape nous avons pu transformer nos str en int ou number grace notamment a TF-IDF mais helas nos etiquettes 'ham' et 'spam' eux sont encore en str or que ce sont nos vatraibles cibles qui sont la

Vu que nous voulons eux aussi les transformer en int ou number donc on va passer en *ENCODE LABEL* qui permet simplement de convertir les catégories textuelles en nombres entiers.

Dans notre problème de classification binaire, nous souhaitons associer :
- "ham" → 0 : message normal, qui n'est pas un spam.
- "spam" → 1 : message indésirable, considéré comme un spam.

In [8]:
encoder = LabelEncoder()

y = encoder.fit_transform(df['processed_message'])

In [9]:
#Sauvegarder nos donnees

# 1. enregistrer  notre encodage
np.save('y_encoded.npy', y)

# 2. enregistrer les tf-idf
scipy.sparse.save_npz('X_features.npz', X)

print("Success! Features and labels saved for the modeling stage.")

Success! Features and labels saved for the modeling stage.


## Qu'avons nous appris??
J'ai appris comment apres la preparations de mes data passer d'un dataset avec des str en int ou number avce les differnts fontions comme le BoW pas tres efficace mais le TF-IDF vient regulariser cela et aussi nos variables importante avec l'encoding
